# Model Experiment Template

Evaluate a candidate reconstruction model on Colab. Latency, GPU usage and the output file are logged to MLflow on Google Drive via `backend/experiments/evaluation/`.

Colab clones the repo from GitHub, so push changes to `evaluation/` before running.

Note: don't run multiple notebooks at once as they share one SQLite database on Drive.

## GPU check

In [ ]:
!nvidia-smi

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0))

## Setup

In [ ]:
import sys
from pathlib import Path
from google.colab import drive

!pip install -q mlflow nvidia-ml-py
![ -d /content/Random-Skier ] || git clone -q https://github.com/Da-Builder/Random-Skier.git /content/Random-Skier
sys.path.append("/content/Random-Skier/backend/experiments")
drive.mount("/content/drive")

from evaluation import Config, evaluate

## Config

In [ ]:
config = Config(
    model_name="template",
    repo="https://github.com/<org>/<model>.git",
)

## Install model

In [ ]:
MODEL_DIR = Path("/content/model")

# Clone the model repo
![ -d {MODEL_DIR} ] || git clone -q {config.repo} {MODEL_DIR}

# TODO: install dependencies following the repo's README
# !pip install -q -r {MODEL_DIR}/requirements.txt
# !pip install -q -e {MODEL_DIR}

# TODO: download weights (and SMPL files if needed) to Drive so they survive runtime resets

## Model

Same shape as `backend/ec2/model.py`, so the class can be moved there unchanged. Load weights in `load()` so load time is measured separately from reconstruction. `reconstruct()` must write a single output file into `out_dir` and return its path: that file is logged as the MLflow artifact and is what `backend/ec2/main.py` uploads.

In [ ]:
class Model:
    def load(self) -> None:
        """Load the model weights onto the GPU."""
        # TODO: load the model weights
        raise NotImplementedError

    def reconstruct(self, video_path: Path, out_dir: Path) -> Path:
        """Reconstruct the skier from a video and return the output file path."""
        # TODO: run the model on video_path and write the result into out_dir
        raise NotImplementedError

## Evaluate

In [ ]:
# Check the error column for clips that failed
evaluate(Model(), config)

## View runs

Locally, with Google Drive for Desktop syncing `random-skier/`:

```bash
mlflow ui --backend-store-uri sqlite:///<path-to-drive>/random-skier/mlflow/mlflow.db
```

Or inside Colab:

In [ ]:
from google.colab import output

!nohup mlflow ui --backend-store-uri sqlite:///{config.mlflow_dir}/mlflow.db --port 5000 > /dev/null 2>&1 &
output.serve_kernel_port_as_window(5000)